# CodeAlpha Task 4: Disease Prediction - Exploratory Data Analysis & Baseline Training

This notebook demonstrates loading the clinical datasets for disease prediction (including Heart Disease, Pima Indians Diabetes, and Breast Cancer), performing exploratory data analysis, mapping feature correlations, and training a baseline XGBoost or Random Forest model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, roc_curve

# Set plotting style
sns.set_theme(style="whitegrid")

## 1. Load Heart Disease Dataset

In [ ]:
# Resolve relative path to datasets
dataset_path = Path("../datasets/disease_prediction_heart.csv")
if dataset_path.exists():
    df = pd.read_csv(dataset_path)
    print(f"Loaded Heart Disease dataset: {df.shape[0]} rows, {df.shape[1]} columns.")
else:
    # Fallback/mock data for notebook testing
    print("Dataset path not found, using synthesized data for notebook demonstration.")
    np.random.seed(42)
    df = pd.DataFrame(np.random.randn(1000, 13), columns=[
        'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal'
    ])
    df['target'] = np.random.choice([0, 1], size=1000, p=[0.55, 0.45])

df.head()

## 2. Exploratory Data Analysis & Feature Correlations

In [ ]:
plt.figure(figsize=(12, 10))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Correlation Matrix of Clinical Disease Features")
plt.show()

## 3. Fit baseline Random Forest Classifier

In [ ]:
# Preprocess data: handle missing values if any
X = df.drop(columns=['target']) if 'target' in df.columns else df.iloc[:, :-1]
y = df['target'] if 'target' in df.columns else df.iloc[:, -1]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

print("Baseline Random Forest Performance:")
print(classification_report(y_test, y_pred))
print(f"ROC AUC Score: {roc_auc_score(y_test, y_prob):.4f}")